# K-Fold Evaluation Notebook (Kaggle)

This notebook evaluates saved fold checkpoints on their corresponding validation splits.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/mruniverse8/kaggle-experiments-.git"
REPO_DIR = Path("/kaggle/working/kaggle-experiments-")
BRANCH = "twitter_sentiment"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repo ready at:", REPO_DIR)


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer

import sys
sys.path.insert(0, str(Path("src").resolve()))

from dz2_causal.dataset import register_special_tokens
from dz2_causal.modeling import CausalExtractionModel
from dz2_causal.eval_utils import evaluate_dataframe_jaccard

CFG = json.loads(Path("config/kaggle_eval_kfold.json").read_text())
CFG


In [ ]:
df = pd.read_csv(CFG["train_csv"]).dropna(subset=["text", "selected_text"]).reset_index(drop=True)
splits = list(
    StratifiedKFold(
        n_splits=CFG["n_splits"],
        shuffle=True,
        random_state=CFG["seed"],
    ).split(df, df["sentiment"])
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = Path(CFG["output_dir"])

fold_results = []
for fold_id, (_, val_idx) in enumerate(splits):
    ckpt_path = output_dir / f"model_fold{fold_id}.pt"
    if not ckpt_path.exists():
        print(f"Skipping fold {fold_id}: checkpoint missing -> {ckpt_path}")
        continue

    tokenizer = AutoTokenizer.from_pretrained(
        CFG["model_name"],
        use_fast=True,
        trust_remote_code=CFG.get("trust_remote_code", False),
    )
    model = CausalExtractionModel(
        model_name=CFG["model_name"],
        trust_remote_code=CFG.get("trust_remote_code", False),
    )
    register_special_tokens(tokenizer, model=model.lm)

    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model_state_dict"], strict=True)
    model.to(device)
    model.eval()

    val_df = df.iloc[val_idx].reset_index(drop=True)
    eval_out = evaluate_dataframe_jaccard(
        df=val_df,
        model=model,
        tokenizer=tokenizer,
        prompt_text=CFG["prompt_text"],
        device=device,
        max_new_tokens=CFG["max_new_tokens"],
    )

    fold_results.append({
        "fold": fold_id,
        "checkpoint": str(ckpt_path),
        "val_jaccard": eval_out["mean_jaccard"],
    })
    print(f"Fold {fold_id} jaccard={eval_out['mean_jaccard']:.4f}")

fold_results


In [ ]:
if fold_results:
    mean_j = float(np.mean([x["val_jaccard"] for x in fold_results]))
    std_j = float(np.std([x["val_jaccard"] for x in fold_results]))
else:
    mean_j, std_j = 0.0, 0.0

summary = {
    "fold_results": fold_results,
    "mean_jaccard": mean_j,
    "std_jaccard": std_j,
}
summary_path = Path(CFG["output_dir"]) / "kfold_eval_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Saved:", summary_path)
summary
